In [2]:
import pandas as pd

# STEP 1: Load the cleaned data from Week 1 (or your Week 2 output)
df = pd.read_pickle('../data/online_retail_cleaned.pkl')
print("Successfully loaded data for Week 3 calculation.")

# STEP 2: The CLTV Calculation Logic Function [cite: 76, 77, 78]
def calculate_cltv(df):
    # Create a Total Revenue column per transaction
    df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']
    
    # Aggregate data by Customer ID to find lifecycle habits [cite: 77]
    customer_df = df.groupby('CustomerID').agg({
        'InvoiceNo': 'nunique',     # Purchase Frequency component
        'TotalRevenue': 'sum',      # Total Monetary Value
        'CohortMonth': 'first'      # Keep segment tracking
    }).rename(columns={'InvoiceNo': 'TotalOrders'})
    
    # Average Order Value (AOV) [cite: 77]
    customer_df['AOV'] = customer_df['TotalRevenue'] / customer_df['TotalOrders']
    
    # Purchase Frequency (PF) per customer [cite: 77]
    total_customers = customer_df.shape[0]
    total_orders = customer_df['TotalOrders'].sum()
    purchase_frequency = total_orders / total_customers
    
    # Calculate Customer Value (CV)
    customer_df['CustomerValue'] = customer_df['AOV'] * purchase_frequency
    
    # Historical CLTV (Assuming a profit margin multiplier, e.g., 0.70) 
    profit_margin = 0.70
    customer_df['CLTV'] = customer_df['CustomerValue'] * profit_margin
    
    # Group by Cohort Segment to see performance metrics [cite: 77]
    cohort_cltv = customer_df.groupby('CohortMonth')['CLTV'].mean().reset_index()
    
    return cohort_cltv


# STEP 3: RUN THE FUNCTION AND FORCE THE OUTPUT TO DISPLAY

cohort_cltv_summary = calculate_cltv(df)

print("\n📈 --- WEEK 3 CLTV SUMMARY BY COHORT ---")
# This will print your final table directly in your notebook window
print(cohort_cltv_summary)

# Save the final summary locally for Week 4 storytelling tasks [cite: 79]
cohort_cltv_summary.to_pickle('../data/cohort_cltv_summary.pkl')
print("\nWeek 3 CLTV calculation saved locally!")

Successfully loaded data for Week 3 calculation.

📈 --- WEEK 3 CLTV SUMMARY BY COHORT ---
   CohortMonth         CLTV
0      2010-12  1165.183217
1      2011-01  1852.628725
2      2011-02  1076.828631
3      2011-03  1086.774960
4      2011-04  1008.885498
5      2011-05  1984.355904
6      2011-06  1148.516829
7      2011-07  1038.466641
8      2011-08  1300.618224
9      2011-09  1286.959567
10     2011-10  1128.102298
11     2011-11  1019.721435
12     2011-12  1370.993244

Week 3 CLTV calculation saved locally!
